In [ ]:
import os
import pandas as pd
import numpy as np

# 1. Thiết lập đường dẫn
PATH_INPUT = "../data/02_interim/pm25_weather.csv"
PATH_OUTPUT = "../data/03_processed/pm25_features_ready.csv"

try:
    # 2. Đọc dữ liệu và sắp xếp thời gian
    df = pd.read_csv(PATH_INPUT)
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.sort_values('datetime').reset_index(drop=True)

    # 3. NHÓM 1: Biến chu kỳ thời gian, mùa 
    df['hour'] = df['datetime'].dt.hour
    df['day_of_week'] = df['datetime'].dt.dayofweek 
    df['month'] = df['datetime'].dt.month
    df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
    
    # Tạo biến Mùa mưa (Từ tháng 5 đến tháng 11)
    df['is_rainy_season'] = df['month'].apply(lambda x: 1 if 5 <= x <= 11 else 0)

    # 4. NHÓM 2: Hàm Traffic Proxy TP.HCM
    def get_hcm_traffic_intensity(hour, is_weekend):
        if is_weekend == 1:
            evening_hangout = np.exp(-0.5 * ((hour - 19.5) / 2.5)**2) 
            return np.clip(evening_hangout * 0.4 + 0.3, 0, 1)
        else:
            morning_peak = np.exp(-0.5 * ((hour - 8.0) / 1.5)**2)
            evening_peak = np.exp(-0.5 * ((hour - 18.25) / 1.7)**2)
            base = 0.35 if 6 <= hour <= 22 else 0.1
            return np.clip(max(morning_peak, evening_peak) + base, 0, 1)

    df['traffic_proxy'] = df.apply(lambda row: get_hcm_traffic_intensity(row['hour'], row['is_weekend']), axis=1)

    # 5. NHÓM 3: Biến Trễ (Lag Features)
    df['pm25_lag1'] = df['pm25'].shift(1)
    df['pm25_lag2'] = df['pm25'].shift(2)
    df['pm25_lag3'] = df['pm25'].shift(3)
    df['pm25_lag24'] = df['pm25'].shift(24)

    df['traffic_lag1'] = df['traffic_proxy'].shift(1)
    df['traffic_lag2'] = df['traffic_proxy'].shift(2)
    df['precip_lag1'] = df['precip'].shift(1)
    df['wind_speed_lag1'] = df['wind_speed'].shift(1)

    # 6. NHÓM 4: Trung bình trượt (Rolling Averages)
    df['pm25_roll3'] = df['pm25'].shift(1).rolling(window=3).mean()
    df['pm25_roll6'] = df['pm25'].shift(1).rolling(window=6).mean()
    df['pm25_roll12'] = df['pm25'].shift(1).rolling(window=12).mean()
    df['pm25_roll24'] = df['pm25'].shift(1).rolling(window=24).mean()

    # 7. Dọn dẹp dữ liệu và bỏ cột không cần thiết
    initial_len = len(df)
    df_clean = df.dropna().reset_index(drop=True)
    df_clean = df_clean.drop(columns=['wind_dir'])
    final_len = len(df_clean)

    # 8. Lưu file
    os.makedirs(os.path.dirname(PATH_OUTPUT), exist_ok=True)
    df_clean.to_csv(PATH_OUTPUT, index=False)

    print("-" * 40)
    print("Đã tạo các đặc trưng cần thiết")
    print(f"Đã loại bỏ {initial_len - final_len} dòng khởi động (NaN).")
    print(f"Dữ liệu đã có {len(df_clean.columns)} đặc trưng.")
    print("-" * 40)

except FileNotFoundError:
    print(f"Không tìm thấy file {PATH_INPUT}.")
except Exception as e:
    print(f"Lỗi: {e}")

# Hiển thị vài dòng để kiểm tra
df_clean.head()

Đang tiến hành Feature Engineering...
----------------------------------------
Đã tạo các đặc trưng cần thiết
Đã loại bỏ 24 dòng khởi động (NaN).
Dữ liệu đã có 24 đặc trưng.
----------------------------------------


,datetime,pm25,temp,humidity,precip,wind_speed,hour,day_of_week,month,is_weekend,...,pm25_lag3,pm25_lag24,traffic_lag1,traffic_lag2,precip_lag1,wind_speed_lag1,pm25_roll3,pm25_roll6,pm25_roll12,pm25_roll24
0,2024-01-02 00:00:00,33.1,26.1,85,0.0,4.0,0,1,1,0,...,51.9,17.6,0.120170,0.437777,0.0,3.3,45.400000,49.500000,41.208333,34.962500
1,2024-01-02 01:00:00,31.1,25.9,87,0.0,5.4,1,1,1,0,...,45.4,17.2,0.100001,0.120170,0.0,4.0,39.133333,48.433333,41.066667,35.608333
2,2024-01-02 02:00:00,32.7,25.5,91,0.0,3.2,2,1,1,0,...,38.9,18.8,0.100019,0.100001,0.0,5.4,34.366667,43.416667,40.958333,36.187500
3,2024-01-02 03:00:00,37.6,25.3,91,0.0,4.1,3,1,1,0,...,33.1,21.8,0.100335,0.100019,0.0,3.2,32.300000,38.850000,41.066667,36.766667
4,2024-01-02 04:00:00,42.1,24.9,94,0.0,4.3,4,1,1,0,...,31.1,24.4,0.103866,0.100335,0.0,4.1,33.800000,36.466667,41.608333,37.425000
